In [1]:
inputs = [35,25]

In [2]:
type(inputs)

list

In [3]:
weights = [0.8, 0.1]

# **Sum Function**

In [4]:
def sum_func(inputs: list, weights: list):
    # res = 0
    # for _input, _weight in zip(inputs, weights):
    #     res += _input * _weight
    return sum(input_ * weight_ for input_, weight_ in zip(inputs, weights))

In [5]:
sum_func(inputs, weights)

30.5

# **Step Function**

In [6]:
def step_func(sum):
    return int(sum >= 1)

In [7]:
s = sum_func(inputs, weights)

In [8]:
step_func(s)

1

# **Using numpy to compute sum effectively**

In [9]:
import numpy as np

In [10]:
def sum_func(inputs, weights):
    return np.array(inputs) @ np.array(weights)

In [11]:
sum_func(inputs, weights)

np.float64(30.5)

## **Gradient Descent: First Attempt at Implementation**

In [12]:
import numpy as np

def grad(features, rows, learning_rate):
    """
    assuming features = no. of features,
    rows is in the shape [[[x1, x2, x3, ...], y], [...], [...]],
    and learning rate is... learning rate :)
    """
    weights = np.random.randn(features)
    bias = 0

    for row in rows:
        x, y = row

        prediction = weights @ x + bias
        error = prediction - y

        djdw = (error) * (x) * 2.0
        djdb = (error) * 2.0

        weights -= djdw * learning_rate
        bias -= djdb * learning_rate
    return weights, bias

In [13]:
# testing with f(x) = 3x + 5

def target_func(x):
    return 3 * x + 5

In [14]:
data = [
    (np.array([x], dtype=float), target_func(x))
    for x in range(1, 100)
]

In [15]:
print(grad(1, data, 0.1))

(array([-9.62501338e+241]), np.float64(-9.722185135805507e+239))


## **Bad results because no epochs and high learning rate**

In [16]:
import numpy as np

def grad(features, rows, learning_rate, epochs=1000):

    weights = np.random.randn(features) * 0.01
    bias = 0.0

    for epoch in range(epochs):

        total_loss = 0

        for x, y in rows:

            prediction = weights @ x + bias
            error = prediction - y

            total_loss += error**2

            djdw = 2 * error * x
            djdb = 2 * error

            weights -= learning_rate * djdw
            bias -= learning_rate * djdb

        if epoch % 100 == 0:
            print("Epoch:", epoch, "MSE:", total_loss / len(rows))

    return weights, bias

In [17]:
# testing with f(x) = 3x + 5

def target_func(x):
    return 3 * x + 5

In [18]:
data = [
    (np.array([x], dtype=float), target_func(x))
    for x in range(1, 100)
]

In [19]:
print(grad(1, data, 0.0001))

Epoch: 0 MSE: 282.4012062002895
Epoch: 100 MSE: 1.7752101961896534
Epoch: 200 MSE: 1.094055706478367
Epoch: 300 MSE: 0.6742626261651199
Epoch: 400 MSE: 0.4155456494134688
Epoch: 500 MSE: 0.2560993002512572
Epoch: 600 MSE: 0.15783308448003064
Epoch: 700 MSE: 0.09727196650689107
Epoch: 800 MSE: 0.05994836570095106
Epoch: 900 MSE: 0.03694596376809948
(array([3.00431647]), np.float64(4.570521586092685))


## **Attempting to apply it to a real dataset**

In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("job_salary_prediction_dataset.csv")

In [21]:
x = df.drop(columns=["salary"])
y = df["salary"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    random_state = 11,
    test_size = 0.2
)

In [22]:
from sklearn.preprocessing import RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector

transformations = ColumnTransformer([
    ("numerical", RobustScaler(), make_column_selector(dtype_include="int64")),
    ("one-hot", OneHotEncoder(drop='first', sparse_output=False), ["job_title", "industry", "remote_work", "location", "company_size"]),
    ("education", OrdinalEncoder(categories=[["High School", "Diploma", "Bachelor", "Master", "PhD"]]), ["education_level"]),
])

In [23]:
x_train = transformations.fit_transform(x_train)

In [24]:
x_train

array([[ 0.4       , -0.6       , -0.33333333, ...,  0.        ,
         1.        ,  1.        ],
       [ 0.8       ,  0.        , -0.33333333, ...,  0.        ,
         0.        ,  2.        ],
       [-0.9       , -0.2       ,  0.66666667, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 0.1       ,  0.        ,  0.66666667, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  1.        , ...,  0.        ,
         0.        ,  2.        ],
       [ 0.7       ,  0.5       , -0.33333333, ...,  0.        ,
         0.        ,  2.        ]], shape=(200000, 39))

In [25]:
x_test = transformations.transform(x_test)

In [26]:
train_data = list(zip(x_train, y_train))

In [27]:
test_data = zip(x_test, y_test)

In [28]:
weights, bias = grad(x_train.shape[1], train_data, learning_rate = 1e-6, epochs = 1000)

Epoch: 0 MSE: 7297300977.219514


KeyboardInterrupt: 

In [ ]:
predictions = x_test @ weights + bias

In [ ]:
predictions.shape

(50000,)

In [ ]:
mse = sum(((y_test - predictions) ** 2)) / x_test.shape[0]

In [ ]:
mse

56642081.339566074

# **Optimizing for larger datasets**

In [29]:
import numpy as np

def grad(features, x, y, learning_rate, epochs=1000):

    weights = np.random.randn(features) * 0.01
    x, y = x.astype(np.float32), y.astype(np.float32)
    bias = 0.0

    for epoch in range(epochs):

        prediction = x @ weights + bias
        error = prediction - y

        djdw = (2 / len(x)) * (x.T @ error)
        djdb = (2 / len(x)) * np.sum(error)

        weights -= learning_rate * djdw
        bias -= learning_rate * djdb
        if epoch % 50 == 0:
            print(epoch, np.mean(error**2))
    return weights, bias

In [30]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("job_salary_prediction_dataset.csv")

In [31]:
x = df.drop(columns=["salary"])
y = df["salary"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    random_state = 11,
    test_size = 0.2
)

In [32]:
from sklearn.preprocessing import RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector

transformations = ColumnTransformer([
    ("numerical", RobustScaler(), make_column_selector(dtype_include="int64")),
    ("one-hot", OneHotEncoder(drop='first', sparse_output=False), ["job_title", "industry", "remote_work", "location", "company_size"]),
    ("education", OrdinalEncoder(categories=[["High School", "Diploma", "Bachelor", "Master", "PhD"]]), ["education_level"]),
])

In [33]:
from sklearn.preprocessing import StandardScaler

y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(
    y_train.to_numpy().reshape(-1,1)
).flatten()

In [34]:
x_train = transformations.fit_transform(x_train)

In [35]:
x_test = transformations.transform(x_test)

In [36]:
weights, bias = grad(x_train.shape[1], x_train, y_train_scaled, learning_rate = 0.008, epochs = 3000)

0 1.0003061033728826
50 0.7597415130949088
100 0.6141792555225409
150 0.5141197062796293
200 0.44160366958225744
250 0.386595244580908
300 0.3432995924421482
350 0.3082411710083311
400 0.2792444098521022
450 0.2548834704403628
500 0.2341793939130792
550 0.21642972052802673
600 0.20111020719953918
650 0.1878164832989307
700 0.17622826722995152
750 0.16608661197007965
800 0.15717886654171767
850 0.1493283393402759
900 0.14238691949966364
950 0.13622962471888947
1000 0.13075044975536973
1050 0.1258591250962334
1100 0.12147853450450004
1150 0.117542624330886
1200 0.11399468970457347
1250 0.1107859560144625
1300 0.10787439596082096
1350 0.1052237372651841
1400 0.10280262646342252
1450 0.10058392163636248
1500 0.0985440924174113
1550 0.09666270976426723
1600 0.09492201118381206
1650 0.09330652961514053
1700 0.09180277618167977
1750 0.09039896864239065
1800 0.089084798691937
1850 0.08785123234464573
1900 0.08669033853498327
1950 0.08559514181458613
2000 0.08455949565078594
2050 0.083577973356

In [37]:
predictions = x_test @ weights + bias   

In [38]:
predictions = y_scaler.inverse_transform(
    predictions.reshape(-1,1)
).flatten()

In [39]:
from sklearn.metrics import mean_absolute_error, r2_score

mean_absolute_error(y_test, predictions)

7377.29218878424

In [40]:
r2_score(y_test, predictions)

0.9285168877213508

# **Classification Model - Logistic Regression**

In [91]:
import numpy as np

# stochastic approach

def grad(features, rows, learning_rate, epochs=1000):

    weights = np.random.randn(features) * 0.01
    bias = 0.0

    sigmoid = lambda x: 1 / (1 + np.e**(-x))

    for epoch in range(epochs):

        total_loss = 0

        for x, y in rows:

            prediction = weights @ x + bias # linear result
            prediction = sigmoid(prediction) # sigmoid squishes into probabilities
            prediction = np.clip(prediction, 1e-15, 1-1e-15) # prevent prediction = 1, sigmoid = -inf, and training error.
            error = prediction - y

            total_loss += -(y*np.log(prediction) + (1-y)*np.log(1-prediction))

            djdw = error * x
            djdb = error

            weights -= learning_rate * djdw
            bias -= learning_rate * djdb

        if epoch % 100 == 0:
            print("Epoch:", epoch, "Cross-Entropy Loss:", total_loss / len(rows))

    return weights, bias

In [92]:
def predict(x, weights, bias, threshold):
    sigmoid = lambda x: 1 / (1 + np.e**(-x))
    return 1 if sigmoid(weights @ x + bias) > threshold else 0

In [93]:
# data: [x1, x2]: if x1 + x2 >= 10, then 1. 0 <= x1, x2. <= 10

data = []

for _ in range(500):
    x1 = np.random.uniform(0, 10)
    x2 = np.random.uniform(0, 10)

    y = int(x1 + x2 >= 10)

    data.append((np.array([x1, x2]), y))

In [94]:
train_size = int(0.8 * 500)

train_data = data[:train_size]
test_data = data[train_size:]

In [95]:
weights, bias = grad(2, train_data, 0.1, 1000)

Epoch: 0 Cross-Entropy Loss: 0.9598955099744865
Epoch: 100 Cross-Entropy Loss: 0.0733714691538688
Epoch: 200 Cross-Entropy Loss: 0.05711768170537205
Epoch: 300 Cross-Entropy Loss: 0.050507128013648754
Epoch: 400 Cross-Entropy Loss: 0.04630478193723111
Epoch: 500 Cross-Entropy Loss: 0.04307205994896482
Epoch: 600 Cross-Entropy Loss: 0.04030203912507498
Epoch: 700 Cross-Entropy Loss: 0.0378704709230945
Epoch: 800 Cross-Entropy Loss: 0.035760931886910055
Epoch: 900 Cross-Entropy Loss: 0.03396776844970826


In [96]:
x_test, y_test = [row[0] for row in test_data], np.array([row[1] for row in test_data])

In [97]:
y_pred = np.array([predict(x, weights, bias, 0.5) for x in x_test])

In [100]:
accuracy = np.mean(y_pred == y_test)

print("Accuracy:", accuracy)

Accuracy: 0.98


In [102]:
weights, bias

(array([6.84249209, 7.14126155]), np.float64(-69.87925262225738))

In [103]:
import numpy as np  

# batch approach

def grad(x, y, learning_rate, epochs=1000):

    features = x.shape[1]

    weights = np.random.randn(features) * 0.01
    bias = 0.0

    x = x.astype(np.float32)
    
    sigmoid = lambda x: 1 / (1 + np.exp(-x))

    for epoch in range(epochs):

            prediction = x @ weights + bias # linear result
            prediction = sigmoid(prediction) # sigmoid squishes into probabilities
            prediction = np.clip(prediction, 1e-15, 1-1e-15) # prevent prediction = 1, sigmoid = -inf, and training error.

            error = prediction - y

            total_loss = -np.mean(y*np.log(prediction) + (1-y)*np.log(1-prediction))

            djdw = (x.T @ error) / x.shape[0]
            djdb = np.mean(error)

            weights -= learning_rate * djdw
            bias -= learning_rate * djdb

            if epoch % 100 == 0:
                        print("Epoch:", epoch, "Cross-Entropy Loss:", total_loss)

    return weights, bias

In [127]:
def predict(x, weights, bias, threshold = 0.5):
    sigmoid = lambda x: 1 / (1 + np.exp(-x))
    return np.array([sigmoid(x @ weights + bias) >= threshold])

In [130]:
# data: [x1, x2]: if x1 + x2 >= 10, then 1. 0 <= x1, x2. <= 10

x = []
y = []

for _ in range(500):
    x1 = np.random.uniform(0, 10)
    x2 = np.random.uniform(0, 10)

    label = int(x1 + x2 >= 10)

    x.append([x1, x2])
    y.append(label)

x = np.array(x)
y = np.array(y)

In [131]:
x.shape, y.shape

((500, 2), (500,))

In [132]:
train_size = int(0.8 * 500)

x_train, y_train, x_test, y_test = x[:train_size], y[:train_size], x[train_size:], y[train_size:]

In [133]:
for arr in [x_train, y_train, x_test, y_test]:
    print(arr.shape)

(400, 2)
(400,)
(100, 2)
(100,)


In [134]:
weights, bias = grad(x_train, y_train, 0.01, 5000)

Epoch: 0 Cross-Entropy Loss: 0.6852506536217183
Epoch: 100 Cross-Entropy Loss: 0.6510925592235864
Epoch: 200 Cross-Entropy Loss: 0.6331448757923454
Epoch: 300 Cross-Entropy Loss: 0.6162873589867368
Epoch: 400 Cross-Entropy Loss: 0.6004443410299722
Epoch: 500 Cross-Entropy Loss: 0.5855456276225429
Epoch: 600 Cross-Entropy Loss: 0.5715246132055429
Epoch: 700 Cross-Entropy Loss: 0.5583185065584051
Epoch: 800 Cross-Entropy Loss: 0.54586845176155
Epoch: 900 Cross-Entropy Loss: 0.5341195435454305
Epoch: 1000 Cross-Entropy Loss: 0.5230207610731231
Epoch: 1100 Cross-Entropy Loss: 0.5125248416241517
Epoch: 1200 Cross-Entropy Loss: 0.5025881123450923
Epoch: 1300 Cross-Entropy Loss: 0.49317029491261677
Epoch: 1400 Cross-Entropy Loss: 0.484234294866264
Epoch: 1500 Cross-Entropy Loss: 0.4757459846438256
Epoch: 1600 Cross-Entropy Loss: 0.4676739870431643
Epoch: 1700 Cross-Entropy Loss: 0.45998946393905693
Epoch: 1800 Cross-Entropy Loss: 0.4526659135699573
Epoch: 1900 Cross-Entropy Loss: 0.4456789785

In [135]:
y_pred = predict(x_test, weights, bias)

In [136]:
accuracy = np.mean(y_pred == y_test)

print("Accuracy:", accuracy)

Accuracy: 0.98


## **Attempting to implement a neural network**

In [17]:
class neuron:

    def __init__(self, weights, bias):
        self.weights = weights
        self.bias = bias
        
    def output(self, x):
        return self.sigmoid(x @ self.weights + self.bias)

    def sigmoid(self, x):   
        return 1 / (1 + np.exp(-x))
    

In [55]:
class network:

    def __init__(self, layer_size, x, y, learning_rate = 0.01, epochs = 1000, threshold = 0.5):
        """
        naming convention
    
            d()d() => partial diff
            ()f => final neuron related
            ()h => hidden neuron related
            j => cost func(cross entropy loss)
            p => sigmoid
            z => linear form(regression equation before segmoid)
            w => weights
            b => bias

        shapes
            x -> (samples, features)
            y -> (samples,)
            hidden_neurons -> (layer_size,)
            hidden_outputs -> (samples, layer_size)
            prediction -> (predictions,) or (samples,)
            error -> (errors,) or (samples,)
            weights -> (features,)
            final neuron weights -> (layer_size,)
            delta_hidden -> (samples, layer_size)
            """  
        features = x.shape[1]
        self.threshold = threshold
        self.hidden_neurons = list([neuron(np.random.randn(features) * 1.0, 0.0) for _ in range(layer_size)])
        self.final_neuron = neuron(np.random.randn(len(self.hidden_neurons)) * 1.0, 0.0)

        for epoch in range(epochs):
            hidden_outputs = self.hidden_outputs(x)
            prediction = self.final_neuron.output(hidden_outputs)
            error = prediction - y

            # djdpf = (error) / (prediction * (1 - prediction))
            # dpfdzf = prediction * (1-prediction)
            # dzfdwf = hidden_outputs.T
    
            djdwf = (hidden_outputs.T @ error) / hidden_outputs.shape[0]
            djdbf = np.mean(error)

            delta_hidden = ((hidden_outputs * (1 - hidden_outputs)) * (self.final_neuron.weights * error[:, None]))

            djdwh = (x.T @ delta_hidden) / x.shape[0]
            djdbh = np.mean(delta_hidden, axis = 0)

            for n in range(len(self.hidden_neurons)):
                 self.hidden_neurons[n].weights -= djdwh[:, n] * learning_rate
                 self.hidden_neurons[n].bias -= djdbh[n] * learning_rate

            self.final_neuron.weights -= djdwf * learning_rate
            self.final_neuron.bias -= djdbf * learning_rate
            
            prediction = np.clip(
                prediction,
                1e-15,
                1-1e-15
            )

            loss = np.mean(
                -(y*np.log(prediction)
                +(1-y)*np.log(1-prediction))
            )

            if epoch % 1000 == 0:
                print(epoch, loss)
                print(djdwh.shape)
                print(djdbh.shape)
                print(np.max(np.abs(djdwf)))
                print(np.max(np.abs(djdwh)))
                 
            
    def hidden_outputs(self, x):
            outputs = np.array(list([n.output(x) for n in self.hidden_neurons]))
            return outputs.T

    def predict_proba(self, x):
        hidden_pred = self.hidden_outputs(x)
        prediction = self.final_neuron.output(hidden_pred)
        return prediction
    def predict(self, x):
        return (self.predict_proba(x) >= self.threshold).astype(np.int32)

In [56]:
x = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=np.float32)

y = np.array([
    0,
    1,
    1,
    0
], dtype=np.float32)

In [57]:
net = network(
    layer_size=4,
    x=x,
    y=y,
    learning_rate=0.1,
    epochs=10000
)

0 0.7190306164681454
(2, 4)
(4,)
0.07673622743316708
0.030958175864383018
1000 0.4512027160288296
(2, 4)
(4,)
0.02813495322602979
0.021553905402013757
2000 0.14703202069079854
(2, 4)
(4,)
0.020001274953723853
0.012830060704907532
3000 0.06317851308285215
(2, 4)
(4,)
0.011004153760987175
0.005514941140120654
4000 0.03713561514157415
(2, 4)
(4,)
0.006980059307322639
0.0031584305790199082
5000 0.025452529749223733
(2, 4)
(4,)
0.004915465994724591
0.0021755974159824317
6000 0.01900452176965095
(2, 4)
(4,)
0.003701032265026318
0.0016425883110855655
7000 0.014979668897997412
(2, 4)
(4,)
0.002918655850671882
0.0013055893273550224
8000 0.012257414620406434
(2, 4)
(4,)
0.002406674405255469
0.0010753459963597915
9000 0.010309285503455
(2, 4)
(4,)
0.002078131896998572
0.0009090953193369905


In [58]:
print(net.predict(x))

[0 1 1 0]


## **This is a single-layer neural network for binary classification**